# Carbon Sequestration, standalone

**Ex-ante ARR carbon removal for one input polygon, in tCO2e.**

Give this notebook a polygon and a project duration. It returns the carbon that restoration could
remove over that duration. It is one slice of the screening tool: the same pathway raster, the
same category and eligibility logic, and the same reference-rate method as component 5.3 of
`F02-P5 Benefit.ipynb`. Everything that is not carbon sequestration is left out: no avoided
emissions, no qualitative benefit profile, no nature or climate context.

**Four steps, and it runs a whole batch of polygons.**

| Step | What it produces | Reuses |
|---|---|---|
| 1. Pathway and eligibility | area and share per pathway, and the categories present in the polygon | logic of 4.1 |
| 2. Restore activities and the carbon gate | the ARR activities that apply, and which of them get a carbon number | logic of 4.2 plus the 5.3 gate |
| 3. Carbon sequestration | tCO2e over the project duration, central plus an indicative range | logic of 5.3 |
| 4. Walker 2022 comparison | the same area's unrealized carbon storage, as a reference ceiling | new |

Step 3 does not depend on Step 1 or Step 2. It reads the rasters itself, exactly as 5.3 does, so
the headline number is produced by one function and the earlier steps only explain where it comes
from. Step 4 depends on Step 3, and only on Step 3: it measures Walker on the exact pixel mask
Step 3 quantified, handed over in memory, so the two numbers always describe the same ground.

Point the notebook at one or more folders of AOI shapefiles and it processes every polygon it
finds. Output stays per AOI, one folder each, plus one summary table across all of them.

## Standalone by design

This notebook does **not** import `config.py` or `common.py`. Every constant and every helper it
needs is written inside it, so the file can be copied out of the repo and run on its own, and so a
reader can audit the whole calculation without opening another file.

> **The constants are a COPY, not a link.** Section A mirrors the `ARR_*`, `PATHWAY_*` and layer
> path settings of `config.py` as they stood on 2026-07-29. If `config.py` changes, this notebook
> does not follow. That is the cost of standalone. Section A ends with a drift check: when the
> notebook happens to sit next to a readable `config.py`, it compares every mirrored constant and
> prints the differences. Run it whenever a number here has to match the main tool. Peatland stays
> excluded here for the same reason it is excluded in 5.3, so the two agree by default.

## What the number is, and is not

Living biomass only, aboveground plus belowground. No soil carbon, no peat soil, no dead wood, no
litter, no avoided emissions. No leakage deduction, no uncertainty deduction, no non-permanence
buffer. The low and high figures are an indicative screening range, not a confidence interval and
not a creditable volume. This is pre-feasibility screening, not project-grade MRV.

Method reference: `NBS-v3-ANX-B` v2 (2026-07-28), Sections 3.2 and 4.

---
## Setup

In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.features import geometry_mask
from rasterio.warp import Resampling, calculate_default_transform, reproject

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

---
## Inputs

Everything editable is in the next cell. No kernel restart is needed: the values are read when the
run cell executes, not at import time.

- **`AOI_FOLDERS`** the folders holding your AOI files. Every `.shp`, `.geojson` and `.gpkg` under
  them is treated as one AOI. Add as many folders as you like.
- **`AOI_FILES`** individual files to add on top of the folders, for one-off polygons.
- **`AOI_RECURSIVE`** whether to look inside subfolders as well.
- **`PROJECT_DURATION_YEARS`** whole years, the crediting period being asked about. The growth
  curve is defined to year 40; years beyond that add nothing and are flagged.
- **`OUTPUT_ROOT`** results go to `OUTPUT_ROOT\<run_id>\`, one folder per AOI. The run id is the
  file stem, so `Sumsel_01.shp` writes to `...\Sumsel_01\`.

Any CRS is accepted. Each polygon is reprojected once to `REFERENCE_CRS` (ESRI:54034, equal area)
so hectares are sound, and multipart input is dissolved into one geometry so overlapping parts are
not counted twice.

**One AOI is not a special case.** A single file in `AOI_FILES` runs exactly the same path as a
folder of fifty, so there is no separate single-polygon mode to keep working.

**What is wired up now.** `D:\REQ\KFF\ARR\Carbon Calculation` with recursion on, which finds
seven shapefiles across four landscape folders: BBS_KK and BBS_PS, BNW_KK, KS_KK and KS_PS,
SBBR_KK and SBBR_PS. Every stem is unique, so no output folder collides. Two things about this set
are worth knowing before reading the results.

The `_KK` and `_PS` files are stored in different coordinate systems, geographic degrees for KK and
World Cylindrical Equal Area for PS. That is handled, both are reprojected to `REFERENCE_CRS`, and
each AOI reports the CRS it arrived in so the conversion is visible rather than assumed.

**`_KK` and `_PS` overlap on the ground.** Their bounding boxes cover much of the same landscape,
so the seven totals are seven separate answers to seven separate questions, not seven slices of one
project. Adding them together would count shared hectares twice. The summary table lists them, it
deliberately does not total them.

In [ ]:
# ---- INPUT 1: where the AOI polygons live ------------------------------------------------
# Every .shp / .geojson / .gpkg found under these folders becomes one AOI. <SET: your folders>
AOI_FOLDERS = [
    r"D:\REQ\KFF\ARR\Carbon Calculation",   # BBS, BNW, KS, SBBBR -> 7 shapefiles, 7 AOIs
]
AOI_FILES = [                      # extra individual files, on top of the folders
]
AOI_RECURSIVE = True               # the shapefiles sit in per-landscape subfolders, so keep this on

# ---- INPUT 2: project duration -----------------------------------------------------------
PROJECT_DURATION_YEARS = 20        # whole years, at least 1

# ---- INPUT 3: output location ------------------------------------------------------------
OUTPUT_ROOT = r"D:\NBSTOOLV3\OUTPUTS"   # one subfolder per AOI, named after the file stem

---
## A. Configuration, mirrored from `config.py`

Everything below is a copy of `config.py` as of 2026-07-29, limited to what the carbon path needs.
Layer paths first, then the pathway raster layout, then the ARR method parameters.

Why each parameter is what it is belongs in `NBS-v3-ANX-B` v2 and in the `config.py` comments;
those comments are kept here so the copy stays readable on its own. Two values are worth repeating
because they move the answer the most:

- **`ARR_BASELINE_CLASS_MGHA` values are PLACEHOLDERS** pending literature references. The result
  scales directly with them.
- **`ARR_STOCKING_ANR` (0.8) is uncalibrated**, doc range 0.7 to 0.85, and the planting versus ANR
  split in `ARR_ANR_PAIRS` is uncalibrated too.

In [ ]:
# ================= MIRRORED FROM config.py (2026-07-29). Keep in sync. =====================
# Change a value here and it changes ONLY this notebook. The drift check below reports any
# difference against a config.py that happens to be importable.

# ---- Reference CRS -----------------------------------------------------------------------
REFERENCE_CRS = "ESRI:54034"       # World Cylindrical Equal Area, locked by the team
M2_PER_HA = 10_000.0

# ---- Layers ------------------------------------------------------------------------------
PATHWAY_RASTER        = r"D:\NBSTOOLV3\SEA_NBS_PATHWAY.tif"                  # 3 bands, v3 layout
AGB_RASTER            = r"D:\NBSTOOLV3\AGBD_GEDI_AEF_pred_SEA_2024.tif"      # Mg/ha, aboveground
ELEVATION_RASTER      = r"D:\NBSTOOLV3\SEA_ELEVATION_54034.tif"              # continuous metres
WORLDCLIM_PREC_RASTER = r"D:\NBSTOOLV3\precipitation_v3.tif"                 # 12 bands, mm/month
WORLDCLIM_MONTHS      = 12
ACTIVITY_TABLE        = r"D:\NBSTOOLV3\canonical_v3_activities.csv"          # Sheet export

# ---- Pathway raster layout, canonical_v3 -------------------------------------------------
# v3, NOT v2: v2 band 2 was a secondary pathway and band 3 the ecosystem. Reading a v2 raster
# with these constants silently swaps ecosystem and cat_code.
PATHWAY_BAND           = 1   # primary pathway, one value per pixel
PATHWAY_ECOSYSTEM_BAND = 2   # reference ecosystem
PATHWAY_CATCODE_BAND   = 3   # 1..17 canonical_v3 category index

PATHWAY_CODES = {0: "No data", 1: "Protect", 2: "Manage", 3: "Restore", 4: "Ineligible"}
RESTORE_CODE = 3                       # the only pathway this notebook quantifies
PATHWAY_ELIGIBLE_CODES = (1, 2, 3)     # code 4 is a screening outcome, not a pathway

PATHWAY_ECOSYSTEM_CODES = {
    0: "None", 1: "Dryland forest", 2: "Mangrove", 3: "Peatland", 4: "Savanna",
}
PATHWAY_ECOSYSTEM_PEATLAND = 3

PATHWAY_CATCODE_LABELS = {
    0: "Mask",
    1: "Cat 1",   2: "Cat 2",   3: "Cat 3A",  4: "Cat 3B",  5: "Cat 4A",  6: "Cat 4B",
    7: "Cat 5",   8: "Cat 6",   9: "Cat 7",  10: "Cat 8A", 11: "Cat 8B", 12: "Cat 8C",
    13: "Cat 9A", 14: "Cat 9B", 15: "Cat 9C", 16: "Cat 9D", 17: "Cat 10",
}
PATHWAY_CATCODE_TO_PATHWAY = {
    1: 1,   # Cat 1  Protect
    2: 4,   # Cat 2  Ineligible
    3: 4,   # Cat 3A Ineligible (savanna)
    4: 3,   # Cat 3B Restore
    5: 4,   # Cat 4A Ineligible (savanna)
    6: 3,   # Cat 4B Restore
    7: 3,   # Cat 5  Restore
    8: 2,   # Cat 6  Manage
    9: 4,   # Cat 7  Ineligible
    10: 4,  # Cat 8A Ineligible (stable natural savanna)
    11: 2,  # Cat 8B Manage
    12: 3,  # Cat 8C Restore
    13: 2,  # Cat 9A Manage
    14: 3,  # Cat 9B Restore
    15: 2,  # Cat 9C Manage
    16: 4,  # Cat 9D Ineligible (settlement)
    17: 3,  # Cat 10 Restore
}
PATHWAY_UNCLASSIFIED_WARN_PCT = 20.0

# ---- ARR carbon sequestration, NBS-v3-ANX-B v2 -------------------------------------------
# Which (cat_code, ecosystem) pairs get carbon quantified. Encodes ANX-B Section 3.2
# "Sequestration calculated". Deliberately NOT the sheet's QB Carbon Sequestration flag: the
# sheet contradicts the method on peat (sheet No, method Yes biomass-only) and savanna (sheet
# Yes, method defers). Savanna (eco 4) absent = deferred; Cat 9B peat (14, 3) absent = rewetting
# only, no planting.
ARR_SEQ_PAIRS = frozenset({
    (4, 1), (4, 2), (4, 3),     # Cat 3B  dryland, mangrove, peat
    (6, 1), (6, 2), (6, 3),     # Cat 4B
    (7, 1), (7, 2), (7, 3),     # Cat 5   (savanna 7,4 excluded)
    (12, 1), (12, 2), (12, 3),  # Cat 8C
    (14, 2),                    # Cat 9B  mangrove only
    (17, 1), (17, 2), (17, 3),  # Cat 10  (savanna 17,4 excluded)
})

# Ecosystems held out of quantification. Activity and benefits still apply, no carbon number.
#   4 savanna  : methodological deferral, carbon is mainly soil and roots, no biomass rates.
#   3 peatland : TEMPORARY team exclusion (2026-07-29). Remove 3 to re-enable peat biomass.
ARR_CARBON_DEFERRED_ECO = frozenset({3, 4})

# Growth phases, Section 4.4. Nothing accrues past year 40.
ARR_YOUNG_END_YEAR = 20
ARR_OLD_END_YEAR = 40

# Reference accumulation rates, Mg DRY MATTER per ha per year, Section 4.6.
ARR_RATE_DM = {                        # keyed on ecosystem code
    2: {"young": 12.0, "old": 7.0},    # mangrove
    3: {"young": 5.7,  "old": 3.5},    # peatland, biomass only
}
ARR_RATE_DM_DRYLAND = {                # keyed on zone code
    1: {"young": 3.4, "old": 2.7},     # humid lowland (rainforest)
    2: {"young": 2.4, "old": 2.0},     # seasonal lowland (conservative, wide range)
    3: {"young": 2.4, "old": 1.9},     # humid montane
}

# Root-to-shoot ratio R (BGB / AGB), Section 4.7, low-biomass classes.
ARR_ROOT_TO_SHOOT = {2: 0.39, 3: 0.25}                    # mangrove, peat
ARR_ROOT_TO_SHOOT_DRYLAND = {1: 0.21, 2: 0.44, 3: 0.32}   # humid lowland, seasonal, montane

# Dryland zone derivation, Section 4.5. Humid montane above 1000 m; else humid lowland if annual
# rainfall above 2000 mm AND fewer than 3 dry months (a dry month is below 100 mm, Walsh 1996);
# else seasonal lowland. Missing data falls to seasonal lowland, the conservative choice.
ARR_DRYLAND_ZONES = {1: "humid lowland", 2: "seasonal lowland", 3: "humid montane"}
ARR_ZONE_ELEV_MONTANE_M = 1000.0
ARR_ZONE_WET_ANNUAL_MM = 2000.0
ARR_ZONE_DRY_MONTH_MM = 100.0
ARR_ZONE_DRY_SEASON_MONTHS = 3
ARR_DRYLAND_DEFAULT_ZONE = 2

# Baseline mode. "class" is the official one; the other two are always reported as diagnostics.
#   "class"         - small assumed standing biomass per current LC state
#   "per_pixel_agb" - the AGB raster per pixel; on vegetated Restore land GEDI reads high and
#                     zeroes the result (Section 4.9), so it is a diagnostic only
#   "none"          - no deduction, gross upper bound
ARR_BASELINE_MODE = "class"
ARR_BASELINE_CLASS_MGHA = {"C4": 25.0, "C5": 5.0, "C6": 0.0}   # PLACEHOLDERS, AGB Mg/ha
ARR_RESTORE_CAT_CSTATE = {                                     # Restore cat_code -> current state
    4: "C4",   # Cat 3B  Forest -> shrub / vegetation
    6: "C5",   # Cat 4B  Forest -> active use
    7: "C6",   # Cat 5   Forest -> barren
    12: "C4",  # Cat 8C  Non-forest -> vegetation
    14: "C5",  # Cat 9B  Non-forest -> active use
    17: "C6",  # Cat 10  Non-forest -> barren
}

# Carbon fraction, dry matter to carbon, Section 4.6.
ARR_CARBON_FRACTION = {1: 0.47, 2: 0.451, 3: 0.47}

# Stocking factor, Section 4.9. ANR/EMR value and the split are UNCALIBRATED.
ARR_STOCKING_PLANTING = 1.0
ARR_STOCKING_ANR = 0.8
ARR_ANR_PAIRS = frozenset({(4, 1), (6, 2), (7, 2), (12, 2), (17, 2)})

# Indicative screening range, Section 4.11. NOT a confidence interval.
ARR_UNCERTAINTY_LOW = 0.7
ARR_UNCERTAINTY_HIGH = 1.2

# ---- Shared conversions ------------------------------------------------------------------
CO2_PER_C = 44.0 / 12.0            # molecular weight ratio, tCO2e per tC
CARBON_COVERAGE_WARN_PCT = 90.0    # flag when a biomass raster covers less than this
# ================= END OF MIRRORED BLOCK ==================================================

# ================= NEW HERE, NOT IN config.py (Step 4) ====================================
# Walker et al. 2022 potential carbon storage. These constants have no counterpart in
# config.py yet, so the drift check does not look at them. Move them into config.py if the
# main tool ever grows a Walker component.
#
# Verified from the file metadata, 2026-08-10:
#   units      Mg CARBON per ha (the "MgCha" in the name), NOT dry matter and NOT CO2e.
#              So the conversion is x 44/12 ONLY. Applying CARBON_FRACTION here would be a
#              double conversion, the same trap as the soil raster in component 3.2.
#   dtype      signed 16-bit integer, nodata -32768
#   grid       MODIS Sinusoidal, 463.31 m pixels, already clipped to a SE Asia polygon
#   range      unrealized 0 to 405 Mg C/ha, current 0 to 2549 Mg C/ha
WALKER_UNR_RASTER = r"D:\NBSTOOLV3\Walker_2022_Carbon_storage_data\Base_Unr_AGB_BGB_SOC_MgCha_500m.tif"
WALKER_CUR_RASTER = r"D:\NBSTOOLV3\Walker_2022_Carbon_storage_data\Base_Cur_AGB_BGB_SOC_MgCha_500m.tif"

# Biomass-only (AGB + BGB, no SOC) unrealized layer. NOT present in the folder as of
# 2026-08-10, which is why the like-for-like column is off. Set the path here and the second
# ceiling column turns on by itself, no other edit needed.
WALKER_UNR_BIOMASS_RASTER = ""

# A stock density, so reprojection must preserve the area-weighted mean.
WALKER_RESAMPLING = "average"
WALKER_POOLS_COMBINED = "AGB + BGB + SOC"
WALKER_POOLS_BIOMASS = "AGB + BGB"
# ===========================================================================================

### Drift check against `config.py`

The cell below is the safeguard that makes the copy above survivable. It imports `config.py` if
one is importable from the working directory, compares every mirrored name, and prints what
differs. It changes nothing: a difference is reported, never applied, because the whole point of
a standalone notebook is that its own values are the ones it used.

No `config.py` in reach is not an error. The notebook is meant to run that way.

The `WALKER_*` constants are deliberately outside the checked list. They are new here and have no
counterpart in `config.py`, so checking them would report a difference on every run and train the
eye to ignore the output.

In [ ]:
# Compare the mirrored block against config.py, if one is importable. Reports only, never patches.
MIRRORED_NAMES = [
    "REFERENCE_CRS", "PATHWAY_RASTER", "AGB_RASTER", "ELEVATION_RASTER",
    "WORLDCLIM_PREC_RASTER", "WORLDCLIM_MONTHS", "ACTIVITY_TABLE",
    "PATHWAY_BAND", "PATHWAY_ECOSYSTEM_BAND", "PATHWAY_CATCODE_BAND", "PATHWAY_CODES",
    "RESTORE_CODE", "PATHWAY_ELIGIBLE_CODES", "PATHWAY_ECOSYSTEM_CODES",
    "PATHWAY_ECOSYSTEM_PEATLAND", "PATHWAY_CATCODE_LABELS", "PATHWAY_CATCODE_TO_PATHWAY",
    "PATHWAY_UNCLASSIFIED_WARN_PCT", "ARR_SEQ_PAIRS", "ARR_CARBON_DEFERRED_ECO",
    "ARR_YOUNG_END_YEAR", "ARR_OLD_END_YEAR", "ARR_RATE_DM", "ARR_RATE_DM_DRYLAND",
    "ARR_ROOT_TO_SHOOT", "ARR_ROOT_TO_SHOOT_DRYLAND", "ARR_DRYLAND_ZONES",
    "ARR_ZONE_ELEV_MONTANE_M", "ARR_ZONE_WET_ANNUAL_MM", "ARR_ZONE_DRY_MONTH_MM",
    "ARR_ZONE_DRY_SEASON_MONTHS", "ARR_DRYLAND_DEFAULT_ZONE", "ARR_BASELINE_MODE",
    "ARR_BASELINE_CLASS_MGHA", "ARR_RESTORE_CAT_CSTATE", "ARR_CARBON_FRACTION",
    "ARR_STOCKING_PLANTING", "ARR_STOCKING_ANR", "ARR_ANR_PAIRS",
    "ARR_UNCERTAINTY_LOW", "ARR_UNCERTAINTY_HIGH", "CO2_PER_C", "CARBON_COVERAGE_WARN_PCT",
]


def check_config_drift(names=MIRRORED_NAMES) -> list[str]:
    """Report differences between the mirrored block and config.py. Returns the difference list."""
    try:
        import importlib
        cfg = importlib.import_module("config")
        importlib.reload(cfg)
    except Exception as exc:
        print(f"No config.py in reach ({type(exc).__name__}), drift check skipped. "
              "This notebook runs on its own values.")
        return []

    here = globals()
    diffs: list[str] = []
    for name in names:
        if name not in here:
            diffs.append(f"{name}: missing from this notebook")
        elif not hasattr(cfg, name):
            diffs.append(f"{name}: not in config.py (config may have dropped or renamed it)")
        elif here[name] != getattr(cfg, name):
            diffs.append(f"{name}: notebook {here[name]!r}  !=  config {getattr(cfg, name)!r}")

    if diffs:
        print(f"DRIFT: {len(diffs)} of {len(names)} mirrored settings differ from config.py.")
        for d in diffs:
            print("  -", d)
        print("Nothing was changed. Copy the config value across by hand if this notebook should "
              "match the main tool.")
    else:
        print(f"No drift: all {len(names)} mirrored settings match config.py.")
    return diffs


config_drift = check_config_drift()

---
## B. Helpers

The parts of `common.py` this notebook needs, copied in: the AOI contract, the raster reader, the
activity table reader, the result container, and a few formatters. Same behaviour as the originals,
so a number produced here is produced the same way as in the main tool.

The one contract worth knowing when reading the code below is **`like=`** on
`load_raster_clipped`. It forces a second raster onto the exact grid of an already loaded one, so
`a.values[i]` and `b.values[i]` describe the same ground. Every read after the first one uses it.
Without it, a pixel's cat_code could be paired with a different pixel's ecosystem, and the whole
join would be silently wrong.

In [ ]:
# ---- AOI contract -------------------------------------------------------------------------

@dataclass(frozen=True)
class AOI:
    """The project area, already in REFERENCE_CRS. `area_ha` is the denominator for any share."""

    geometry: gpd.GeoSeries
    area_ha: float
    source_crs: str


def prepare_aoi(polygon) -> AOI:
    """Normalise the polygon once. Accepts a path, a GeoDataFrame, or a GeoSeries, in any CRS.

    Reprojects to REFERENCE_CRS, dissolves multipart input into one geometry so overlapping parts
    are not counted twice, and measures the area.
    """
    if isinstance(polygon, (str, Path)):
        polygon = gpd.read_file(str(polygon))
    geoms = polygon.geometry if isinstance(polygon, gpd.GeoDataFrame) else polygon
    if geoms.crs is None:
        raise ValueError("The polygon has no CRS. Area cannot be measured without one.")
    # Short form ("EPSG:4326") when the CRS has an authority code, full WKT otherwise. Reporting
    # only: nothing downstream parses it.
    source_crs = geoms.crs.to_string()

    geoms = geoms.to_crs(REFERENCE_CRS)
    dissolved = gpd.GeoSeries([geoms.union_all()], crs=REFERENCE_CRS)
    area_ha = float(dissolved.area.sum()) / M2_PER_HA
    if area_ha <= 0:
        raise ValueError("The polygon has zero area after reprojection.")
    return AOI(geometry=dissolved, area_ha=area_ha, source_crs=source_crs)


# ---- Raster access ------------------------------------------------------------------------

_RESAMPLING = {
    "nearest": Resampling.nearest,     # categorical layers, and any layer whose min or max is read
    "bilinear": Resampling.bilinear,   # continuous layer read only for its mean
    "average": Resampling.average,     # stock and probability layers, preserves the area mean
}


@dataclass(frozen=True)
class RasterSlice:
    """A raster clipped and reprojected to the AOI. Masked cells are nodata or outside the polygon."""

    values: np.ma.MaskedArray
    pixel_area_ha: float
    transform: object = None
    crs: object = None

    @property
    def valid_count(self) -> int:
        return int(self.values.count())

    @property
    def valid_area_ha(self) -> float:
        return self.valid_count * self.pixel_area_ha


def load_raster_clipped(path: str, aoi: AOI, resampling: str = "nearest", band: int = 1,
                        like: "RasterSlice | None" = None) -> RasterSlice:
    """Clip `path` to the AOI, reproject to REFERENCE_CRS, mask to the polygon.

    `like` forces the output onto exactly the grid of an already loaded slice, which is what makes
    a per-pixel join between two rasters meaningful. Returns an all-masked array when the raster
    does not cover the AOI; that is handled as "not applicable", not as an error.
    """
    rs = _RESAMPLING[resampling]
    geom = aoi.geometry.iloc[0]

    with rasterio.open(path) as src:
        if like is not None:
            dst_transform = like.transform
            dst_h, dst_w = like.values.shape
        else:
            # Source resolution carried into REFERENCE_CRS, cropped to the AOI bounding box, so
            # only the AOI window is warped instead of the whole SEA raster.
            base_transform, _, _ = calculate_default_transform(
                src.crs, REFERENCE_CRS, src.width, src.height, *src.bounds
            )
            minx, miny, maxx, maxy = geom.bounds
            inv = ~base_transform
            c0, r0 = inv * (minx, maxy)
            c1, r1 = inv * (maxx, miny)
            col_off, row_off = int(np.floor(c0)), int(np.floor(r0))
            dst_w = max(1, int(np.ceil(c1)) - col_off)
            dst_h = max(1, int(np.ceil(r1)) - row_off)
            dst_transform = base_transform * Affine.translation(col_off, row_off)

        # NaN fill separates "no source coverage" from a genuine 0 in a categorical raster.
        fill = float(src.nodata) if src.nodata is not None else np.nan
        dst = np.full((dst_h, dst_w), fill, dtype="float64")
        reproject(
            source=rasterio.band(src, band), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_transform, dst_crs=REFERENCE_CRS,
            src_nodata=src.nodata, dst_nodata=fill, resampling=rs,
        )
        src_nodata = src.nodata

    outside = geometry_mask([geom.__geo_interface__], out_shape=(dst_h, dst_w),
                            transform=dst_transform, invert=False)
    nodata_mask = np.isnan(dst) if src_nodata is None else (dst == src_nodata)
    values = np.ma.masked_array(dst, mask=(outside | nodata_mask))
    pixel_area_ha = abs(dst_transform.a * dst_transform.e) / M2_PER_HA
    return RasterSlice(values=values, pixel_area_ha=pixel_area_ha,
                       transform=dst_transform, crs=REFERENCE_CRS)


# ---- Activity catalog ---------------------------------------------------------------------

_ACTIVITY_COL_ALIASES = {
    "cat_id": "cat_code", "cat_code": "cat_code",
    "ecosystem": "ecosystem",
    "activity id": "activity_id", "activity_id": "activity_id",
    "activity": "activity",
    "benefit nature": "benefit_nature", "benefit_nature": "benefit_nature",
    "benefit people": "benefit_people", "benefit_people": "benefit_people",
    "benefit climate": "benefit_climate", "benefit_climate": "benefit_climate",
    "qb avoided emissions": "qb_avoided", "qb_avoided": "qb_avoided",
    "qb carbon sequestration": "qb_sequestration", "qb_sequestration": "qb_sequestration",
}
_ECOSYSTEM_NAME_TO_CODE = {"dryland forest": 1, "mangrove": 2, "peatland": 3, "savanna": 4}


def load_activity_table(path: str) -> dict[tuple[int, int], list[dict]]:
    """Load canonical_v3_activities, keyed on (cat_code, ecosystem).

    Reads the Sheet export headers directly. Rows with a blank Ecosystem are the ineligible
    categories: they carry no join key and are skipped, and Step 2 handles them through
    PATHWAY_CATCODE_TO_PATHWAY instead.
    """
    df = pd.read_csv(path)
    df.columns = [_ACTIVITY_COL_ALIASES.get(c.strip().lower(), c.strip().lower())
                  for c in df.columns]
    required = {"cat_code", "ecosystem", "activity_id", "activity", "benefit_nature",
                "benefit_people", "benefit_climate", "qb_avoided", "qb_sequestration"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns {sorted(missing)}.")

    def as_bool(v) -> bool:
        return str(v).strip().lower() in {"yes", "y", "true", "1"}

    def eco_code(v):
        s = str(v).strip()
        return int(s) if s.isdigit() else _ECOSYSTEM_NAME_TO_CODE.get(s.lower())

    def clean_id(v) -> str:
        if pd.isna(v):
            return ""
        try:
            return str(int(float(v)))       # "11.0" from a float column -> "11"
        except (ValueError, TypeError):
            return str(v).strip()

    def clean_text(v) -> str:
        return "" if pd.isna(v) else " ".join(str(v).split())

    table: dict[tuple[int, int], list[dict]] = {}
    for _, r in df.iterrows():
        if pd.isna(r["ecosystem"]) or str(r["ecosystem"]).strip() == "":
            continue
        ec = eco_code(r["ecosystem"])
        if ec is None:
            raise ValueError(f"Unknown ecosystem {r['ecosystem']!r} in {path}.")
        table.setdefault((int(r["cat_code"]), ec), []).append({
            "activity_id": clean_id(r["activity_id"]),
            "activity": clean_text(r["activity"]),
            "benefit_nature": clean_text(r["benefit_nature"]),
            "benefit_people": clean_text(r["benefit_people"]),
            "benefit_climate": clean_text(r["benefit_climate"]),
            "qb_avoided": as_bool(r["qb_avoided"]),
            "qb_sequestration": as_bool(r["qb_sequestration"]),
        })
    return table


def resolve_activity_table_path(path: str = ACTIVITY_TABLE) -> str | None:
    """The configured catalog path, or a copy sitting next to the notebook, or None."""
    if Path(path).exists():
        return path
    local = Path.cwd() / "canonical_v3_activities.csv"
    return str(local) if local.exists() else None


# ---- Result container and formatters ------------------------------------------------------

@dataclass
class StepResult:
    """Common shape returned by each step. Same fields as ComponentResult in the main tool.

    `masks` and `grid` are the in-memory handoff from Step 3 to Step 4: the boolean pixel masks
    Step 3 worked on and the RasterSlice defining their grid. They are numpy and affine objects,
    never JSON, so `to_jsonable` drops them, the same way the main tool drops `rasters`. They
    exist so Step 4 measures Walker on exactly the pixels Step 3 quantified instead of
    reconstructing the mask and risking a different answer.
    """

    component: str
    applicable: bool
    narrative: str
    tables: dict = field(default_factory=dict)
    values: dict = field(default_factory=dict)
    flags: list = field(default_factory=list)
    masks: dict = field(default_factory=dict)
    grid: object = None


def not_applicable(component: str, reason: str) -> StepResult:
    return StepResult(component=component, applicable=False, narrative=reason)


def safe_pct(part: float, whole: float) -> float:
    """Share in percent, 0.0 when the denominator is zero. Keeps output free of NaN."""
    return 0.0 if whole <= 0 else part / whole * 100.0


def fmt_ha(value: float) -> str:
    return f"{value:,.0f} ha"


def sort_by_area(rows: list) -> list:
    return sorted(rows, key=lambda r: r.area_ha, reverse=True)


def as_frame(rows: list) -> pd.DataFrame:
    """Any table (dataclasses or dicts) as a DataFrame, for display and for CSV."""
    recs = [asdict(x) if is_dataclass(x) and not isinstance(x, type) else x for x in rows]
    return pd.DataFrame(recs)


def show(r: StepResult, tables: bool = True) -> None:
    """Print one step: header, narrative, tables, flags. `values` is left to the caller."""
    from IPython.display import display

    print(f"[{r.component}]" + ("" if r.applicable else "  (not applicable)"))
    if r.narrative:
        print(" ", r.narrative)
    if tables:
        for name, tbl in r.tables.items():
            if tbl:
                print(f"  {name}:")
                display(as_frame(tbl))
    for f in r.flags:
        print("FLAG:", f)


def to_jsonable(obj):
    """Convert a result tree into plain JSON types: dataclasses, sets, numpy scalars and arrays."""
    if is_dataclass(obj) and not isinstance(obj, type):
        # 'masks' and 'grid' hold numpy arrays and an affine grid, not JSON. They are a
        # step-to-step handoff, not part of the saved contract.
        return {k: to_jsonable(v) for k, v in asdict(obj).items()
                if k not in ("masks", "grid")}
    if isinstance(obj, (set, frozenset)):
        return sorted(obj, key=str)
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

### Find the AOI files

Scans `AOI_FOLDERS` and adds `AOI_FILES`, and reports what it found. Nothing is read yet: this only
builds the work list, so a typo in a path is caught here rather than half way through a batch.

Duplicates are removed by resolved path, and the list is sorted by name so a rerun processes the
AOIs in the same order. A shapefile's sidecar files (`.dbf`, `.shx`, and the rest) are not listed
separately: only the `.shp` is.

In [ ]:
AOI_SUFFIXES = (".shp", ".geojson", ".gpkg")


def discover_aois(folders=None, files=None, recursive=True) -> list[Path]:
    """The AOI work list: every vector file under `folders`, plus `files`. Sorted, deduplicated.

    A missing folder raises rather than being skipped quietly. A silent skip would turn a typo
    into "that AOI has no Restore area", which looks like a result instead of a mistake.
    """
    folders = AOI_FOLDERS if folders is None else folders
    files = AOI_FILES if files is None else files

    found: list[Path] = []
    for folder in folders:
        root = Path(folder)
        if not root.is_dir():
            raise FileNotFoundError(f"AOI folder not found: {folder}")
        for suffix in AOI_SUFFIXES:
            found += root.rglob(f"*{suffix}") if recursive else root.glob(f"*{suffix}")
    for f in files:
        path = Path(f)
        if not path.exists():
            raise FileNotFoundError(f"AOI file not found: {f}")
        found.append(path)

    seen: dict[str, Path] = {}
    for path in found:
        seen.setdefault(str(path.resolve()).lower(), path)
    return sorted(seen.values(), key=lambda p: (p.stem.lower(), str(p).lower()))


aoi_paths = discover_aois(recursive=AOI_RECURSIVE)
if not aoi_paths:
    raise ValueError("No AOI files found. Set AOI_FOLDERS or AOI_FILES in the input cell.")

print(f"{len(aoi_paths)} AOI file(s) found, {PROJECT_DURATION_YEARS} year duration, "
      f"output under {OUTPUT_ROOT}")
for path in aoi_paths:
    print(f"  {path.stem:30s}  {path}")

run_ids = [p.stem for p in aoi_paths]
if len(set(run_ids)) != len(run_ids):
    dupes = sorted({r for r in run_ids if run_ids.count(r) > 1})
    raise ValueError(
        f"Two AOI files share a name: {dupes}. The run id is the file stem and it names the "
        "output folder, so one would overwrite the other. Rename one of them."
    )

---
## Step 1. Pathway and eligibility

Same reading of the pathway raster as component 4.1: how much of the polygon falls under each
pathway, and which canonical_v3 categories and reference ecosystems are present.

**Data.** `PATHWAY_RASTER`, all three bands. Band 1 is the pathway (0 no data, 1 Protect, 2 Manage,
3 Restore, 4 Ineligible), band 2 the reference ecosystem, band 3 the 1..17 category index.

**Decisions locked.**

- The denominator is the **whole polygon area**, not the covered part. Pixels with no pathway value
  are absorbed into an `Unclassified` row, so the table sums to 100 percent of the site and a
  poorly covered polygon is visible instead of hidden.
- Code 4 `Ineligible` is a screening outcome in the same band, not a pathway. It is tabulated but
  is not counted in `eligible_ha`.
- Only the **Restore** row matters for the carbon steps. Protect and Manage are shown because a
  reader needs to see what share of the site the carbon number does not cover.

**Downstream use.** Context only. Step 3 rereads the raster itself, so nothing here feeds the
carbon calculation. The Restore area shown here is the pool that Step 2 and Step 3 narrow down.

**Open item.** A polygon that straddles the edge of the pathway raster gets a large `Unclassified`
share; the flag reports it above 20 percent, but there is no automatic rejection.

In [ ]:
# Step 1. Pathway distribution and the categories present. Logic of component 4.1.

@dataclass(frozen=True)
class PathwayShare:
    """One row of the pathway breakdown."""

    code: int
    label: str
    area_ha: float
    pct: float          # share of the TOTAL polygon area, not of the classified part
    is_pathway: bool    # False for Ineligible and Unclassified


UNCLASSIFIED_LABEL = "Unclassified"


def _codes_present(aoi: AOI, band: int, labels: dict, like=None) -> dict:
    """Which codes occur in one band, with labels and areas. Code 0 is mask, not a category."""
    raster = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=band, like=like)
    codes, counts = np.unique(raster.values.compressed(), return_counts=True)
    area_by_code = {int(c): int(n) * raster.pixel_area_ha
                    for c, n in zip(codes.tolist(), counts.tolist()) if int(c) != 0}
    ordered = sorted(area_by_code)
    return {
        "codes": ordered,
        "labels": [labels.get(c, f"Unknown code {c}") for c in ordered],
        "area_ha": {labels.get(c, f"Unknown code {c}"): area_by_code[c] for c in ordered},
    }


def analyze_pathway_distribution(aoi: AOI) -> StepResult:
    """Step 1. Area and share of the polygon per primary pathway, canonical_v3."""
    component = "Step 1  Pathway and eligibility"
    primary = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND)

    if primary.valid_area_ha <= 0:
        return not_applicable(
            component,
            "The pathway layer does not cover this polygon, so no pathway can be recommended and "
            "no carbon can be estimated.",
        )

    rows: list[PathwayShare] = []
    classified_ha = 0.0
    for code, label in PATHWAY_CODES.items():
        if code == 0:
            continue  # folded into Unclassified below, together with nodata
        area_ha = int((primary.values == code).sum()) * primary.pixel_area_ha
        classified_ha += area_ha
        rows.append(PathwayShare(code=code, label=label, area_ha=area_ha,
                                 pct=safe_pct(area_ha, aoi.area_ha),
                                 is_pathway=code in PATHWAY_ELIGIBLE_CODES))

    unclassified_ha = max(0.0, aoi.area_ha - classified_ha)
    rows.append(PathwayShare(code=0, label=UNCLASSIFIED_LABEL, area_ha=unclassified_ha,
                             pct=safe_pct(unclassified_ha, aoi.area_ha), is_pathway=False))

    eligible_ha = sum(r.area_ha for r in rows if r.is_pathway)
    restore_ha = next((r.area_ha for r in rows if r.code == RESTORE_CODE), 0.0)

    flags: list[str] = []
    unclassified_pct = safe_pct(unclassified_ha, aoi.area_ha)
    if unclassified_pct > PATHWAY_UNCLASSIFIED_WARN_PCT:
        flags.append(
            f"Step 1: {unclassified_pct:.0f}% of the polygon carries no pathway value. Every "
            "share below describes only the remainder of the site."
        )
    if restore_ha <= 0:
        flags.append(
            "Step 1: no Restore area in this polygon. Steps 2 and 3 will find nothing to quantify."
        )

    ecosystem = _codes_present(aoi, PATHWAY_ECOSYSTEM_BAND, PATHWAY_ECOSYSTEM_CODES, like=primary)
    catcode = _codes_present(aoi, PATHWAY_CATCODE_BAND, PATHWAY_CATCODE_LABELS, like=primary)
    pathway_rows = sort_by_area([r for r in rows if r.is_pathway and r.area_ha > 0])

    return StepResult(
        component=component,
        applicable=True,
        narrative="",   # the table is the output; downstream reads values
        tables={"pathway_distribution": rows},
        values={
            "total_area_ha": aoi.area_ha,
            "eligible_ha": eligible_ha,
            "eligible_pct": safe_pct(eligible_ha, aoi.area_ha),
            "restore_ha": restore_ha,
            "restore_pct": safe_pct(restore_ha, aoi.area_ha),
            "pathway_ha": {r.label: r.area_ha for r in rows if r.is_pathway},
            "dominant_pathway": pathway_rows[0].label if pathway_rows else None,
            "unclassified_pct": unclassified_pct,
            "reference_ecosystem_codes": ecosystem["codes"],
            "reference_ecosystem_labels": ecosystem["labels"],
            "reference_ecosystem_area_ha": ecosystem["area_ha"],
            "cat_code_codes": catcode["codes"],
            "cat_code_labels": catcode["labels"],
        },
        flags=flags,
    )

---
## Step 2. Restore activities and the carbon gate

Same join as component 4.2, plus the gate that decides which of those categories reach Step 3.
Its job is to make the carbon number traceable: every hectare in the Step 3 total appears here as
a category with a named activity, and every Restore hectare that is left out appears here with the
reason.

**Data.** Bands 3 and 2 of the pathway raster on **one shared grid**, cross-tabulated into the
unique `(cat_code, ecosystem)` pairs present with their area. That is the joint distribution, not
the two marginals of Step 1, so Cat 5 on dryland and Cat 5 on peat stay separate rows. Each pair is
joined to `canonical_v3_activities`.

**Decisions locked.**

- **The gate is `ARR_SEQ_PAIRS`, not the sheet's `QB Carbon Sequestration` column.** The sheet
  currently disagrees with the method on peat (sheet No, method quantifies peat biomass) and on
  savanna (sheet Yes, method defers). The doc wins. The sheet flag is still displayed in the table
  as `qb_seq_sheet`, so the disagreement stays visible until the sheet is reconciled.
- `arr_carbon_status` is the **single** gate function. Step 3 calls the same function, so the
  status shown here cannot drift from what Step 3 actually quantifies.
- A pair with no catalog row is **flagged, not dropped**. Silence about a missing row would read
  as "no activity applies here", which is a different statement.
- Ineligible categories (pathway 4) carry no activity by design and are shown with that note.

**Example render.**

> | pathway | category | ecosystem | area_ha | activity | arr_carbon |
> | Restore | Cat 5 | Dryland forest | 4,120 | Active reforestation | quantified |
> | Restore | Cat 4B | Peatland | 900 | Peat revegetation | deferred, peatland excluded |

**Downstream use.** Explanation only. Step 3 does not read this step, by the same design choice
that lets 5.3 run without the 4.2 JSON.

**Open item.** The activity catalog is a CSV export of the Sheet. If the CSV is not at
`ACTIVITY_TABLE` and not next to the notebook, this step degrades to categories and areas without
activity names, and says so, rather than failing.

In [ ]:
# Step 2. The activities that apply, and which categories reach the carbon step.
# Logic of component 4.2, plus the ARR gate that Step 3 uses.


def arr_carbon_status(cat_code: int, ecosystem: int) -> tuple[bool, str]:
    """The single ARR carbon gate. Returns (quantified, reason).

    Step 2 shows it and Step 3 obeys it, so the two cannot disagree. Encodes ANX-B Section 3.2
    plus the ecosystems currently held out.
    """
    if PATHWAY_CATCODE_TO_PATHWAY.get(cat_code) != RESTORE_CODE:
        return False, "not Restore"
    # Ecosystem deferral is tested first so savanna reads as a deferral, not as a missing pair.
    if ecosystem in ARR_CARBON_DEFERRED_ECO:
        eco = PATHWAY_ECOSYSTEM_CODES.get(ecosystem, str(ecosystem)).lower()
        why = "temporarily excluded" if ecosystem == PATHWAY_ECOSYSTEM_PEATLAND else "deferred"
        return False, f"{eco} {why}"
    if (cat_code, ecosystem) not in ARR_SEQ_PAIRS:
        return False, "no ARR sequestration for this category and ecosystem"
    return True, "quantified"


def analyze_activity_list(aoi: AOI) -> StepResult:
    """Step 2. Activities per (cat_code, ecosystem) present, with the carbon gate on each."""
    component = "Step 2  Restore activities and the carbon gate"

    # Bands 3 and 2 on one shared grid, so each pixel pairs its own cat_code with its ecosystem.
    catcode = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                                  band=PATHWAY_CATCODE_BAND)
    ecosystem = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                                    band=PATHWAY_ECOSYSTEM_BAND, like=catcode)

    cat, eco = catcode.values, ecosystem.values
    valid = ~np.ma.getmaskarray(cat) & ~np.ma.getmaskarray(eco)
    if not valid.any():
        return not_applicable(
            component, "The pathway layer does not cover this polygon, so no activities apply."
        )

    pairs = np.stack([cat[valid].astype(int), eco[valid].astype(int)], axis=1)
    uniq, counts = np.unique(pairs, axis=0, return_counts=True)
    px_area = catcode.pixel_area_ha

    flags: list[str] = []
    table_path = resolve_activity_table_path()
    if table_path is None:
        table: dict = {}
        flags.append(
            f"Step 2: the activity catalog was not found at {ACTIVITY_TABLE} and not next to the "
            "notebook. Categories and areas are still listed, activity names are not. Step 3 is "
            "unaffected: it does not read the catalog."
        )
    else:
        table = load_activity_table(table_path)

    rows: list[dict] = []
    by_category: dict[str, dict] = {}
    quantified_ha = 0.0
    excluded_ha: dict[str, float] = {}

    for idx in np.argsort(counts)[::-1]:            # dominant categories first
        cc, ec = int(uniq[idx][0]), int(uniq[idx][1])
        if cc == 0:
            continue                                 # mask
        area_ha = int(counts[idx]) * px_area
        pathway_code = PATHWAY_CATCODE_TO_PATHWAY.get(cc)
        pathway = PATHWAY_CODES.get(pathway_code, "Unknown")
        cat_label = PATHWAY_CATCODE_LABELS.get(cc, f"cat {cc}")
        eco_label = PATHWAY_ECOSYSTEM_CODES.get(ec, f"ecosystem {ec}")
        will_count, reason = arr_carbon_status(cc, ec)

        if pathway_code == RESTORE_CODE:
            if will_count:
                quantified_ha += area_ha
            else:
                excluded_ha[reason] = excluded_ha.get(reason, 0.0) + area_ha

        if ec == 0:
            acts = []
            flags.append(
                f"Step 2: {cat_label} appears with ecosystem 0 (no reference) on "
                f"{fmt_ha(area_ha)}; the pathway script should have masked these pixels."
            )
        elif pathway_code == 4:
            acts = []                                # Ineligible: no activity by design
        else:
            acts = table.get((cc, ec), [])
            if not acts and table:
                flags.append(
                    f"Step 2: no catalog row for ({cat_label}, {eco_label}); {fmt_ha(area_ha)} "
                    "left without an activity."
                )

        base = {"pathway": pathway, "category": cat_label, "ecosystem": eco_label,
                "area_ha": round(area_ha, 1)}
        if acts:
            for a in acts:
                rows.append({**base, "activity_id": a["activity_id"], "activity": a["activity"],
                             "qb_seq_sheet": "Yes" if a["qb_sequestration"] else "No",
                             "arr_carbon": reason})
        else:
            note = ("(ineligible, no activity)" if pathway_code == 4
                    else "(catalog not loaded)" if not table else "(no catalog match)")
            rows.append({**base, "activity_id": "", "activity": note,
                         "qb_seq_sheet": "", "arr_carbon": reason})

        by_category[f"{cat_label} | {eco_label}"] = {
            "pathway": pathway, "cat_code": cc, "ecosystem": ec, "area_ha": area_ha,
            "arr_carbon_quantified": will_count, "arr_carbon_reason": reason,
            "activities": acts,
        }

    # Where the catalog and the method disagree, on Restore rows only.
    disagree = [r for r in rows
                if r["pathway"] == PATHWAY_CODES[RESTORE_CODE] and r["qb_seq_sheet"]
                and (r["qb_seq_sheet"] == "Yes") != (r["arr_carbon"] == "quantified")]
    if disagree:
        flags.append(
            f"Step 2: {len(disagree)} Restore row(s) where the sheet's QB Carbon Sequestration "
            "flag and the ANX-B gate disagree. The gate wins; the sheet needs reconciling. "
            + "; ".join(f"{r['category']} {r['ecosystem']} (sheet {r['qb_seq_sheet']}, "
                        f"gate {r['arr_carbon']})" for r in disagree)
        )

    restore_rows = [r for r in rows if r["pathway"] == PATHWAY_CODES[RESTORE_CODE]]
    return StepResult(
        component=component,
        applicable=True,
        narrative="",
        tables={"activities": rows,
                "restore_activities": restore_rows},   # the subset the carbon step cares about
        values={
            "by_category": by_category,
            "category_count": len(by_category),
            "activity_count": sum(1 for r in rows if r["activity_id"]),
            "arr_quantified_ha": quantified_ha,        # must match Step 3's quantified_ha
            "arr_excluded_ha": excluded_ha,            # Restore area left out, by reason
            "activity_table_used": table_path or "",
        },
        flags=flags,
    )

---
## Step 3. Carbon sequestration

The headline number. Identical in method and in parameters to component 5.3 of `F02-P5 Benefit`,
so the same polygon and the same duration give the same total in both places.

**Data.** The pathway raster (bands 1, 2, 3) and the AGB raster, all on one shared grid. Dryland
also reads the elevation raster and the 12-band monthly precipitation raster to derive its growth
zone. No dependency on Steps 1 and 2, and no dependency on the activity catalog.

**Method, per hectare (ANX-B Section 4).**

```
AGB_cum  = rate_young * years_young + rate_old * years_old   # Mg d.m./ha, Y1-20 then Y21-40
TB_cum   = AGB_cum * (1 + R)                                 # add belowground, R = root:shoot
CO2e_cum = TB_cum * CF * 44/12                               # CF = carbon fraction
Net      = max(0, CO2e_cum - baseline_CO2e)                  # deduct biomass already on site
Net_adj  = Net * stocking_factor                             # planting 1.0, ANR/EMR 0.8
Total    = sum(Net_adj * pixel_area) over eligible pixels
low/high = Total * 0.7 / Total * 1.2
```

**Decisions locked / deferred.**

- **Gate.** `arr_carbon_status`, the same function Step 2 displays. Restore categories 3B, 4B, 5,
  8C, 9B, 10 on Dryland or Mangrove. Peatland is temporarily excluded by team decision
  (2026-07-29); the method and the rates exist, remove 3 from `ARR_CARBON_DEFERRED_ECO` to switch
  it back on. Savanna is deferred on method grounds: its carbon sits in soil and roots, outside the
  biomass scope. Cat 9B on peat is rewetting with no planting in any case.
- **Dryland zone is derived per pixel** (Section 4.5): humid montane above 1000 m; else humid
  lowland if annual rainfall is above 2000 mm and there are fewer than 3 dry months (a dry month is
  below 100 mm, Walsh 1996); else seasonal lowland. Elevation wins over rainfall. Boundary and
  missing-data pixels fall to seasonal lowland, the conservative choice, and are flagged. One
  category can therefore carry up to three different rates across the polygon.
- **Baseline is class-based** (team decision, 2026-07-29). A small assumed standing biomass per
  current land-cover state (`ARR_BASELINE_CLASS_MGHA`: C4 25, C5 5, C6 0 Mg/ha), converted the same
  way as the accumulation and clamped at zero. It replaced the per-pixel AGB baseline, which zeroed
  almost all vegetated Restore land because GEDI over-reads standing biomass at low biomass
  (Section 4.9). Being eligible for Restore does not mean having no biomass, so accumulating from
  zero and then subtracting the current stock is the wrong subtraction. **The class values are
  placeholders pending references**, and the total scales with them. The per-pixel GEDI baseline
  and the gross figure are still computed and reported for comparison.
- **Nodata AGB counts as zero baseline**, which over-credits. Coverage is measured and flagged.
- **Nothing accrues past year 40.** A longer duration is flagged.
- **No leakage, no uncertainty deduction, no non-permanence buffer.**

**Example render.**

> Restoring the eligible areas of this project could remove an estimated 128,000 tCO2e over 20
> years, with an indicative range of 90,000 to 154,000 tCO2e.

**Narrative not yet specified.** Placeholder wording, same as 5.3; replace once the team settles it.

**Open items.** The C4 / C5 / C6 baseline values need references. The ANR and EMR stocking factor
(0.8) and the planting versus ANR split are uncalibrated. The units of the elevation and
precipitation rasters still need verifying.

In [ ]:
# Step 3. Ex-ante ARR carbon removal. Same method and parameters as component 5.3.


@dataclass(frozen=True)
class ArrGroup:
    """One (cat_code, ecosystem, zone) group of restoring pixels."""

    cat_code: int
    ecosystem: int
    category_label: str
    ecosystem_label: str
    zone_label: str           # dryland zone name, or "" for mangrove and peat
    activity_mode: str        # "planting" or "ANR/EMR"
    stocking_factor: float
    area_ha: float
    net_tco2e: float          # central, after baseline and stocking
    net_tco2e_per_ha: float


@dataclass(frozen=True)
class ArrYear:
    """One year of the cumulative removal curve, summed over the polygon."""

    year: int
    cumulative_tco2e: float


def _arr_params(ecosystem: int, zone: int | None):
    """(rate dict, R, CF) for one ecosystem. Dryland reads by zone; the others by ecosystem."""
    if ecosystem == 1:
        return (ARR_RATE_DM_DRYLAND[zone], ARR_ROOT_TO_SHOOT_DRYLAND[zone],
                ARR_CARBON_FRACTION[1])
    return ARR_RATE_DM[ecosystem], ARR_ROOT_TO_SHOOT[ecosystem], ARR_CARBON_FRACTION[ecosystem]


def _arr_accum_co2e_per_ha(rate: dict, r: float, cf: float, years: int) -> float:
    """Cumulative removal per ha before baseline and stocking, tCO2e/ha, at `years`.

    Two growth phases: Young to year 20, Old to year 40. Nothing accrues beyond year 40.
    """
    young = min(years, ARR_YOUNG_END_YEAR)
    old = max(0, min(years, ARR_OLD_END_YEAR) - ARR_YOUNG_END_YEAR)
    agb_dm = rate["young"] * young + rate["old"] * old        # Mg d.m./ha
    tb_dm = agb_dm * (1.0 + r)                                 # add belowground
    return tb_dm * cf * CO2_PER_C


def _arr_dryland_zone(aoi: AOI, like_slice):
    """Per-pixel dryland zone code aligned to `like_slice`, plus a validity mask.

    Section 4.5: humid montane above 1000 m; else humid lowland if annual rainfall above 2000 mm
    and fewer than 3 dry months; else seasonal lowland. Missing inputs get the default zone.
    """
    elev = load_raster_clipped(ELEVATION_RASTER, aoi, resampling="bilinear", like=like_slice)
    prec = [
        load_raster_clipped(WORLDCLIM_PREC_RASTER, aoi, resampling="average", band=b,
                            like=like_slice)
        for b in range(1, WORLDCLIM_MONTHS + 1)
    ]
    prec_stack = np.ma.stack([p.values for p in prec])              # (12, H, W)
    annual = prec_stack.sum(axis=0)                                  # mm/yr
    dry_months = (prec_stack < ARR_ZONE_DRY_MONTH_MM).sum(axis=0)    # count of dry months

    inputs_valid = ~np.ma.getmaskarray(elev.values) & ~np.ma.getmaskarray(annual)
    elev_f = elev.values.filled(0.0)
    annual_f = np.ma.filled(annual, 0.0)                             # missing -> 0 -> not humid
    dry_f = np.ma.filled(dry_months, ARR_ZONE_DRY_SEASON_MONTHS)     # missing -> not humid

    zone = np.full(elev_f.shape, ARR_DRYLAND_DEFAULT_ZONE, dtype=int)
    humid = (annual_f > ARR_ZONE_WET_ANNUAL_MM) & (dry_f < ARR_ZONE_DRY_SEASON_MONTHS)
    zone[humid] = 1
    zone[elev_f > ARR_ZONE_ELEV_MONTANE_M] = 3   # elevation criterion wins over rainfall
    zone[~inputs_valid] = ARR_DRYLAND_DEFAULT_ZONE
    return zone, inputs_valid


def analyze_carbon_sequestration(aoi: AOI, duration_years: int) -> StepResult:
    """Step 3. Ex-ante carbon removal from ARR restoration on the Restore area, in tCO2e."""
    component = "Step 3  Carbon Sequestration (ARR, ex-ante)"

    if duration_years < 1:
        raise ValueError("PROJECT_DURATION_YEARS must be a whole number of years, at least 1.")

    pathway = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND)
    eco = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                              band=PATHWAY_ECOSYSTEM_BAND, like=pathway)
    cat = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                              band=PATHWAY_CATCODE_BAND, like=pathway)
    agb = load_raster_clipped(AGB_RASTER, aoi, resampling="average", like=pathway)
    pix = pathway.pixel_area_ha

    restore = (pathway.values == RESTORE_CODE).filled(False)
    if not restore.any():
        return not_applicable(
            component,
            "No area of this project falls under the Restore pathway, so ARR carbon removal "
            "cannot be estimated.",
        )

    catv = cat.values.filled(0).astype(int)
    ecov = eco.values.filled(0).astype(int)
    agbv = agb.values.filled(0.0).astype(float)
    agb_valid = ~np.ma.getmaskarray(agb.values)

    # Dryland zone, derived once and only if a dryland Restore pixel exists.
    if (restore & (ecov == 1)).any():
        zone_arr, zone_valid = _arr_dryland_zone(aoi, pathway)
    else:
        zone_arr = np.full(restore.shape, ARR_DRYLAND_DEFAULT_ZONE, dtype=int)
        zone_valid = np.ones(restore.shape, dtype=bool)

    groups: list[ArrGroup] = []
    quantified = np.zeros_like(restore, dtype=bool)
    annual = np.zeros(duration_years, dtype=float)
    agb_valid_quant_ha = 0.0
    gross_tco2e = 0.0               # before any baseline deduction, diagnostic
    clamped_ha = 0.0                # area where the GEDI baseline alone would zero the result
    net_classbaseline_tco2e = 0.0   # diagnostic: net under the class-based baseline
    net_gedi_tco2e = 0.0            # diagnostic: net under the per-pixel GEDI baseline

    for cc, ec in sorted(ARR_SEQ_PAIRS):
        if not arr_carbon_status(cc, ec)[0]:
            continue                 # same gate Step 2 displays
        base_mask = restore & (catv == cc) & (ecov == ec)
        if not base_mask.any():
            continue
        # Dryland splits into its three zones; the other ecosystems are a single group.
        subgroups = ([(z, base_mask & (zone_arr == z)) for z in ARR_DRYLAND_ZONES]
                     if ec == 1 else [(None, base_mask)])

        for zone_code, gmask in subgroups:
            n = int(gmask.sum())
            if n == 0:
                continue
            quantified |= gmask
            area = n * pix
            rate, r, cf = _arr_params(ec, zone_code)
            conv = (1.0 + r) * cf * CO2_PER_C
            is_anr = (cc, ec) in ARR_ANR_PAIRS
            stocking = ARR_STOCKING_ANR if is_anr else ARR_STOCKING_PLANTING
            mode = "ANR/EMR" if is_anr else "planting"
            accum = _arr_accum_co2e_per_ha(rate, r, cf, duration_years)

            # Three baselines. ARR_BASELINE_MODE picks the primary; the other two are reported
            # for comparison. base_class and base_none are constant per pixel.
            cstate = ARR_RESTORE_CAT_CSTATE.get(cc)
            base_gedi = agbv[gmask] * conv                                  # per-pixel vector
            base_class = np.full(n, ARR_BASELINE_CLASS_MGHA.get(cstate, 0.0) * conv)
            base_primary = {"class": base_class, "per_pixel_agb": base_gedi,
                            "none": np.zeros(n)}[ARR_BASELINE_MODE]

            def _net(b, a=accum, st=stocking):
                return float(np.maximum(0.0, a - b).sum()) * st * pix

            net = _net(base_primary)
            gross_tco2e += accum * stocking * area                          # baseline = 0
            net_gedi_tco2e += _net(base_gedi)
            net_classbaseline_tco2e += _net(base_class)
            clamped_ha += int((base_gedi >= accum).sum()) * pix             # GEDI diagnostic

            groups.append(ArrGroup(
                cat_code=cc, ecosystem=ec,
                category_label=PATHWAY_CATCODE_LABELS.get(cc, f"cat {cc}"),
                ecosystem_label=PATHWAY_ECOSYSTEM_CODES.get(ec, f"eco {ec}"),
                zone_label=(ARR_DRYLAND_ZONES[zone_code] if ec == 1 else ""),
                activity_mode=mode, stocking_factor=stocking,
                area_ha=area, net_tco2e=net,
                net_tco2e_per_ha=(net / area if area else 0.0),
            ))

            for i, t in enumerate(range(1, duration_years + 1)):
                accum_t = _arr_accum_co2e_per_ha(rate, r, cf, t)
                annual[i] += float(np.maximum(0.0, accum_t - base_primary).sum()) * stocking * pix

            agb_valid_quant_ha += int((gmask & agb_valid).sum()) * pix

    if not groups:
        return not_applicable(
            component,
            "No Restore area carries an ARR activity eligible for carbon sequestration (planting "
            "or ANR on dryland or mangrove), so ex-ante carbon removal cannot be estimated.",
        )

    total = sum(g.net_tco2e for g in groups)
    quantified_ha = sum(g.area_ha for g in groups)

    # Restore area that carries an activity but no carbon number.
    deferred = restore & ~quantified
    savanna_ha = int((deferred & (ecov == 4)).sum()) * pix
    peat_ha = int((deferred & (ecov == PATHWAY_ECOSYSTEM_PEATLAND)).sum()) * pix
    other_deferred_ha = int(deferred.sum()) * pix - savanna_ha - peat_ha
    zone_defaulted_ha = int((quantified & (ecov == 1) & ~zone_valid).sum()) * pix

    flags: list[str] = []
    cov = safe_pct(agb_valid_quant_ha, quantified_ha)
    if ARR_BASELINE_MODE == "per_pixel_agb" and cov < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"Step 3: the AGB raster covers only {cov:.0f}% of the quantified area. Missing AGB "
            "is read as zero baseline, so no standing biomass is deducted there and the removal "
            "is over-estimated by an unknown amount."
        )
    flags.append(
        f"Step 3: baseline mode is '{ARR_BASELINE_MODE}'. The class-based values are PLACEHOLDERS "
        f"(C4 {ARR_BASELINE_CLASS_MGHA['C4']:g}, C5 {ARR_BASELINE_CLASS_MGHA['C5']:g}, "
        f"C6 {ARR_BASELINE_CLASS_MGHA['C6']:g} Mg/ha) pending references, so the total is "
        f"indicative. For comparison, the per-pixel GEDI baseline gives {net_gedi_tco2e:,.0f} "
        f"tCO2e and gross (no baseline) {gross_tco2e:,.0f}."
    )
    if savanna_ha > 0:
        flags.append(
            f"Step 3: {fmt_ha(savanna_ha)} of Restore area is savanna, whose carbon is deferred "
            "(recovery is mainly soil and roots). Activity and benefits still apply."
        )
    if peat_ha > 0:
        flags.append(
            f"Step 3: {fmt_ha(peat_ha)} of Restore area is peatland, temporarily excluded from "
            "carbon quantification by team decision (the biomass method and rates exist; remove "
            "peatland from ARR_CARBON_DEFERRED_ECO to re-enable). Activity and benefits still "
            "apply."
        )
    if zone_defaulted_ha > 0:
        flags.append(
            f"Step 3: {fmt_ha(zone_defaulted_ha)} of dryland could not be zoned (missing "
            "elevation or precipitation) and defaulted to the seasonal-lowland rate."
        )
    if duration_years > ARR_OLD_END_YEAR:
        flags.append(
            f"Step 3: the accumulation curve is defined only to year {ARR_OLD_END_YEAR}; the "
            f"{duration_years - ARR_OLD_END_YEAR} years beyond it add no further removal."
        )
    flags.append(
        "Step 3: the ANR/EMR stocking factor (0.8) and the planting-vs-ANR split are "
        "uncalibrated. Dryland zones use elevation and 12-band monthly precipitation (units to "
        "verify), with a dry month defined as below 100 mm (Walsh 1996)."
    )

    total_low = total * ARR_UNCERTAINTY_LOW
    total_high = total * ARR_UNCERTAINTY_HIGH

    net_by_ecosystem: dict[str, float] = {}
    area_by_ecosystem: dict[str, float] = {}
    area_by_dryland_zone: dict[str, float] = {}
    for g in groups:
        net_by_ecosystem[g.ecosystem_label] = net_by_ecosystem.get(g.ecosystem_label, 0.0) + g.net_tco2e
        area_by_ecosystem[g.ecosystem_label] = area_by_ecosystem.get(g.ecosystem_label, 0.0) + g.area_ha
        if g.zone_label:
            area_by_dryland_zone[g.zone_label] = area_by_dryland_zone.get(g.zone_label, 0.0) + g.area_ha

    # Placeholder wording, see the markdown cell.
    narrative = (
        f"Restoring the eligible areas of this project could remove an estimated {total:,.0f} "
        f"tCO2e over {duration_years} years, with an indicative range of {total_low:,.0f} to "
        f"{total_high:,.0f} tCO2e."
    )

    return StepResult(
        component=component,
        applicable=True,
        narrative=narrative,
        tables={
            "annual_projection": [ArrYear(t + 1, annual[t]) for t in range(duration_years)],
            "groups": sorted(groups, key=lambda g: -g.net_tco2e),
        },
        values={
            "total_tco2e": total,                    # headline central estimate
            "total_low_tco2e": total_low,
            "total_high_tco2e": total_high,
            "total_tco2e_per_ha": (total / quantified_ha if quantified_ha else 0.0),
            "duration_years": duration_years,
            "quantified_ha": quantified_ha,
            "baseline_mode": ARR_BASELINE_MODE,
            "total_gross_tco2e": gross_tco2e,
            "net_perpixel_gedi_tco2e": net_gedi_tco2e,
            "net_classbaseline_tco2e": net_classbaseline_tco2e,
            "gedi_baseline_zeroed_ha": clamped_ha,   # diagnostic on the GEDI baseline
            "net_by_ecosystem_tco2e": net_by_ecosystem,
            "area_by_ecosystem_ha": area_by_ecosystem,
            "area_by_dryland_zone_ha": area_by_dryland_zone,
            "agb_coverage_pct": cov,
            "zone_defaulted_ha": zone_defaulted_ha,
            "deferred_savanna_ha": savanna_ha,
            "deferred_peat_ha": peat_ha,
            "peat_excluded": PATHWAY_ECOSYSTEM_PEATLAND in ARR_CARBON_DEFERRED_ECO,
            "deferred_other_ha": other_deferred_ha,
            "method": "reference-rate / yield-curve, NBS-v3-ANX-B v2",
            "pools_included": ["aboveground biomass", "belowground biomass"],
            "pools_excluded": ["soil organic carbon", "peat soil", "dead wood", "litter",
                               "avoided emissions"],
            "uncertainty": {"low": ARR_UNCERTAINTY_LOW, "high": ARR_UNCERTAINTY_HIGH},
        },
        flags=flags,
        # In-memory handoff to Step 4, dropped before the result is saved. Step 4 must measure
        # Walker on THESE pixels, not on a mask it rebuilds for itself.
        masks={"quantified": quantified, "restore": restore},
        grid=pathway,
    )

---
## Step 4. Walker 2022 comparison

A second, completely independent number for the same ground, so the ARR estimate can be read
against something rather than in a vacuum. Walker et al. 2022 mapped, for every 500 m pixel on
land, the carbon it stores now and the carbon it could store without human land use. The
difference, the **unrealized potential**, is the gap our restoration is trying to close.

**Data.** `Base_Unr_AGB_BGB_SOC_MgCha_500m.tif` (unrealized) and, for context,
`Base_Cur_AGB_BGB_SOC_MgCha_500m.tif` (current storage). Verified from the file metadata on
2026-08-10: units are **Mg CARBON per ha**, signed 16-bit integers, nodata -32768, MODIS
Sinusoidal at 463.31 m, already clipped to a South East Asia polygon. Both are read onto the exact
grid Step 3 used, with `average` resampling because they are stock densities.

**The single most important line of this step**

```
tCO2e = Mg C/ha * area_ha * 44/12          # carbon fraction is NOT applied
```

The layer is already carbon, not dry matter. Multiplying by `CARBON_FRACTION` (0.47) as well would
under-report by half, the same double-conversion trap the soil raster in component 3.2 carries.

**What it compares, and what it cannot.** The ceiling is measured on **exactly the pixels Step 3
quantified**, handed over as a mask in memory, so the two figures cover the same footprint rather
than two different ones.

**Presented as absolute figures side by side** (team choice, 2026-08-10): total tCO2e and tCO2e per
ha for each row, no percentage in the table and none in the sentence. The share of ceiling is still
computed and kept in `values` (`share_of_combined_ceiling_pct`, `share_of_biomass_ceiling_pct`) for
anyone who wants it, and the ceiling test below still runs, because it is a correctness signal
rather than a presentation choice.

Four mismatches survive, and none of them can be fixed by arithmetic. They are printed as flags on
every run, because a reader who takes the ratio at face value will read it wrong.

1. **Pools.** Walker's layer is AGB + BGB + **SOC**. Our estimate is biomass only: soil carbon and
   peat soil are explicitly out of scope. So the denominator contains a pool the numerator never
   claims, and the ratio is understated by construction. Walker released per-pool layers, but the
   folder only holds the combined ones, so the like-for-like column is off. Set
   `WALKER_UNR_BIOMASS_RASTER` to an AGB + BGB layer and the second ceiling column appears with no
   other change.
2. **Time.** Unrealized potential is an equilibrium: the carbon present once recovery has run its
   full course, on the order of centuries. Our number is a flow over `PROJECT_DURATION_YEARS`.
   Comparing them is comparing a distance with a speed, which is why the output is a percentage of
   the ceiling reached in N years and never a difference.
3. **Scenario.** Walker's baseline potential assumes human land use is removed. On cropland and
   settlement that means abandonment, not a project. Our ARR estimate assumes an intervention on
   land already screened as Restore.
4. **Resolution.** 463 m pixels resampled onto a finer grid. A Walker pixel straddling the AOI
   boundary spreads its value across the edge, so small or thin AOIs carry more of this error.

**The check worth having.** If the N-year estimate exceeds the full-recovery ceiling on the same
pixels, something is wrong: the rates, the stocking, or the baseline. That comparison is raised as
a flag, and it is the main reason this step earns its place.

**Example render.**

> | metric | pools | total_tco2e | per_ha_tco2e |
> | ARR estimate, 20 yr | AGB + BGB | 128,000 | 89 |
> | Unrealized ceiling | AGB + BGB | 410,000 | 285 |
> | Unrealized ceiling | AGB + BGB + SOC | 940,000 | 653 |
> | Current storage (context) | AGB + BGB + SOC | 720,000 | 500 |

The two ceiling rows are the point: one is like-for-like against a biomass-only estimate, the other
is the full land ceiling including soil. Reading only the second one understates what the project
achieves.

**Open item.** Ask the team for the per-pool Walker layers. Until they arrive the headline ratio
compares biomass against biomass plus soil, and everyone reading it has to remember that.

In [ ]:
# Step 4. The same pixels measured against Walker et al. 2022 unrealized carbon storage.


@dataclass(frozen=True)
class CeilingRow:
    """One reference figure for the compared area."""

    metric: str
    pools: str
    total_tco2e: float
    per_ha_tco2e: float
    coverage_pct: float       # share of the compared area the layer actually covers


def _walker_tco2e(path: str, aoi: AOI, grid, mask: np.ndarray, area_ha: float):
    """Sum one Walker layer over `mask`. Returns (total tCO2e, per ha, coverage pct).

    The layer is Mg CARBON per ha, so the only conversion is 44/12. Uncovered pixels contribute
    nothing and are reported as coverage, never filled with a guess.
    """
    layer = load_raster_clipped(path, aoi, resampling=WALKER_RESAMPLING, like=grid)
    valid = ~np.ma.getmaskarray(layer.values) & mask
    mgc_per_ha = np.ma.filled(layer.values, 0.0)[valid]
    total = float(mgc_per_ha.sum()) * grid.pixel_area_ha * CO2_PER_C
    covered_ha = int(valid.sum()) * grid.pixel_area_ha
    return total, (total / area_ha if area_ha else 0.0), safe_pct(covered_ha, area_ha)


def analyze_walker_comparison(aoi: AOI, arr: StepResult) -> StepResult:
    """Step 4. Walker 2022 unrealized storage on the pixels Step 3 quantified."""
    component = "Step 4  Walker 2022 comparison"

    if not arr.applicable or "quantified" not in arr.masks:
        return not_applicable(
            component,
            "Step 3 quantified no area, so there is nothing to compare against the Walker "
            "reference.",
        )

    grid = arr.grid
    mask = arr.masks["quantified"]
    area_ha = arr.values["quantified_ha"]
    ours = arr.values["total_tco2e"]
    years = arr.values["duration_years"]

    rows: list[CeilingRow] = [CeilingRow(
        metric=f"ARR estimate, {years} yr", pools=WALKER_POOLS_BIOMASS,
        total_tco2e=ours, per_ha_tco2e=(ours / area_ha if area_ha else 0.0),
        coverage_pct=arr.values["agb_coverage_pct"],
    )]
    flags: list[str] = []
    values: dict = {
        "compared_area_ha": area_ha,
        "compared_on": "the pixels Step 3 quantified",
        "arr_total_tco2e": ours,
        "duration_years": years,
        "walker_units_in": "Mg C/ha",
        "walker_conversion": "x 44/12 only, carbon fraction NOT applied",
    }

    # Ceiling 1: the combined layer, the only one present today.
    unr, unr_ha, unr_cov = _walker_tco2e(WALKER_UNR_RASTER, aoi, grid, mask, area_ha)
    rows.append(CeilingRow(metric="Unrealized ceiling", pools=WALKER_POOLS_COMBINED,
                           total_tco2e=unr, per_ha_tco2e=unr_ha, coverage_pct=unr_cov))
    values["unrealized_combined_tco2e"] = unr
    values["unrealized_combined_per_ha_tco2e"] = unr_ha
    values["share_of_combined_ceiling_pct"] = safe_pct(ours, unr)
    values["walker_coverage_pct"] = unr_cov

    # Ceiling 2: biomass only, like for like. Appears when the layer is configured.
    if WALKER_UNR_BIOMASS_RASTER:
        bio, bio_ha, bio_cov = _walker_tco2e(WALKER_UNR_BIOMASS_RASTER, aoi, grid, mask, area_ha)
        rows.append(CeilingRow(metric="Unrealized ceiling", pools=WALKER_POOLS_BIOMASS,
                               total_tco2e=bio, per_ha_tco2e=bio_ha, coverage_pct=bio_cov))
        values["unrealized_biomass_tco2e"] = bio
        values["unrealized_biomass_per_ha_tco2e"] = bio_ha
        values["share_of_biomass_ceiling_pct"] = safe_pct(ours, bio)
        headline_share = values["share_of_biomass_ceiling_pct"]
        headline_pools = WALKER_POOLS_BIOMASS
    else:
        values["unrealized_biomass_tco2e"] = None
        values["share_of_biomass_ceiling_pct"] = None
        headline_share = values["share_of_combined_ceiling_pct"]
        headline_pools = WALKER_POOLS_COMBINED
        flags.append(
            "Step 4: POOL MISMATCH. The only Walker layer available includes SOC "
            f"({WALKER_POOLS_COMBINED}), while the ARR estimate is biomass only "
            f"({WALKER_POOLS_BIOMASS}). The share of ceiling is therefore understated by "
            "construction, and by an amount that is large on peat and small on dry mineral soil. "
            "Set WALKER_UNR_BIOMASS_RASTER to a biomass-only layer for a like-for-like ratio."
        )

    # Context: how much carbon is already there, same pixels, same pools as the ceiling.
    if WALKER_CUR_RASTER:
        cur, cur_ha, cur_cov = _walker_tco2e(WALKER_CUR_RASTER, aoi, grid, mask, area_ha)
        rows.append(CeilingRow(metric="Current storage (context)", pools=WALKER_POOLS_COMBINED,
                               total_tco2e=cur, per_ha_tco2e=cur_ha, coverage_pct=cur_cov))
        values["current_combined_tco2e"] = cur
        values["current_combined_per_ha_tco2e"] = cur_ha

    values["headline_share_pct"] = headline_share
    values["headline_ceiling_pools"] = headline_pools

    # The check that earns this step its place.
    if unr > 0 and ours > unr:
        flags.append(
            f"Step 4: the {years} year ARR estimate ({ours:,.0f} tCO2e) EXCEEDS the full-recovery "
            f"unrealized ceiling on the same pixels ({unr:,.0f} tCO2e), even though that ceiling "
            "also contains soil carbon. A flow over a few decades cannot beat the equilibrium gap. "
            "Check the accumulation rates, the stocking factor, and the class baseline before "
            "reporting either number."
        )
    if unr_cov < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"Step 4: the Walker layer covers only {unr_cov:.0f}% of the compared area. The "
            "ceiling is the sum over the covered part, so it is too low and the share of ceiling "
            "too high."
        )
    flags.append(
        "Step 4: TIME BASIS. Unrealized potential is an equilibrium reached over centuries; the "
        f"ARR figure is a {years} year flow. Read the ratio as progress toward a ceiling, never "
        "as a shortfall to be made up."
    )
    flags.append(
        "Step 4: SCENARIO. Walker's potential assumes human land use is removed, which on "
        "cropland and settlement means abandonment rather than a project. Resolution is 463 m "
        "resampled onto a finer grid, so boundary pixels smear."
    )

    # Absolute figures side by side, no percentage (team choice, 2026-08-10). The share stays in
    # `values` for anyone who wants it.
    ceiling_txt = " and ".join(
        f"{r.total_tco2e:,.0f} tCO2e ({r.pools})"
        for r in rows if r.metric == "Unrealized ceiling"
    )
    narrative = (
        f"Over {years} years the estimated removal is {ours:,.0f} tCO2e on {fmt_ha(area_ha)}, "
        f"against an unrealized carbon storage of {ceiling_txt} that Walker et al. 2022 map for "
        "the same area."
    )

    return StepResult(component=component, applicable=True, narrative=narrative,
                      tables={"comparison": rows}, values=values, flags=flags)

---
## Run every AOI

One function, `run_one`, does the whole chain for a single polygon and writes that polygon's
output folder. The cell after it walks the work list.

**Per AOI output**, unchanged from the single-polygon version: `OUTPUT_ROOT\<run_id>\` holding
`<run_id>__carbon-sequestration.json` and one CSV per table under `tables\`.

**Failure is isolated.** One bad polygon, a missing sidecar file, a geometry with no CRS, must not
lose the other forty results. Each AOI is wrapped: the error is recorded in the summary row and the
batch carries on. A failed AOI is visible as a row with a message, not as a gap in the table.

**The consistency check runs per AOI.** Step 2 and Step 3 count the eligible area independently,
one by joining categories, the other by masking pixels. They call the same gate function, so the
two areas must agree. A mismatch is recorded on the summary row rather than printed and forgotten.

In [ ]:
# The whole chain for one polygon, plus the per-AOI save.


def check_area_consistency(results: dict) -> str:
    """"OK", "n/a", or a description. Step 2's eligible area must equal Step 3's quantified area."""
    if "2" not in results or "3" not in results:
        return "n/a"
    if not (results["2"].applicable and results["3"].applicable):
        return "n/a"
    a2 = results["2"].values["arr_quantified_ha"]
    a3 = results["3"].values["quantified_ha"]
    tol = max(1e-6, 1e-4 * max(a2, a3))
    return "OK" if abs(a2 - a3) <= tol else f"MISMATCH {a2:,.1f} vs {a3:,.1f} ha"


def save_run(results: dict, aoi: AOI, run_id: str, source: str) -> Path:
    """Write one AOI's JSON and table CSVs. Returns the JSON path."""
    out_dir = Path(OUTPUT_ROOT) / run_id
    tables_dir = out_dir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        "run_id": run_id,
        "stage": "carbon-sequestration",
        "polygon": {"source": source, "area_ha": aoi.area_ha, "source_crs": aoi.source_crs,
                    "reference_crs": REFERENCE_CRS},
        "inputs": {"project_duration_years": PROJECT_DURATION_YEARS},
        # The parameters that produced the number, so a saved run stays reproducible even after
        # config.py moves on.
        "parameters": {n: to_jsonable(globals()[n]) for n in MIRRORED_NAMES if n in globals()},
        "walker": {n: to_jsonable(globals()[n]) for n in
                   ("WALKER_UNR_RASTER", "WALKER_CUR_RASTER", "WALKER_UNR_BIOMASS_RASTER",
                    "WALKER_RESAMPLING")},
        "config_drift": config_drift,
        "area_check": check_area_consistency(results),
        "steps": {k: to_jsonable(v) for k, v in results.items()},
    }
    json_path = out_dir / f"{run_id}__carbon-sequestration.json"
    json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    for key, res in sorted(results.items()):
        for name, tbl in res.tables.items():
            if tbl:
                as_frame(tbl).to_csv(tables_dir / f"{key}_{name}.csv", index=False)
    return json_path


def run_one(path: Path, duration_years: int = None) -> dict:
    """Steps 1 to 4 for one polygon, saved to its own folder. Returns a record for the summary."""
    duration_years = PROJECT_DURATION_YEARS if duration_years is None else duration_years
    run_id = path.stem
    aoi = prepare_aoi(str(path))

    results: dict[str, StepResult] = {}
    results["1"] = analyze_pathway_distribution(aoi)
    results["2"] = analyze_activity_list(aoi)
    results["3"] = analyze_carbon_sequestration(aoi, duration_years)
    results["4"] = analyze_walker_comparison(aoi, results["3"])

    json_path = save_run(results, aoi, run_id, str(path))
    v1, v3, v4 = results["1"].values, results["3"].values, results["4"].values

    return {
        "run_id": run_id,
        "aoi": aoi,
        "results": results,
        "json_path": json_path,
        "row": {
            "run_id": run_id,
            "area_ha": round(aoi.area_ha, 1),
            "restore_ha": round(v1.get("restore_ha", 0.0), 1),
            "quantified_ha": round(v3.get("quantified_ha", 0.0), 1),
            "arr_tco2e": round(v3.get("total_tco2e", 0.0)),
            "arr_low_tco2e": round(v3.get("total_low_tco2e", 0.0)),
            "arr_high_tco2e": round(v3.get("total_high_tco2e", 0.0)),
            "arr_tco2e_per_ha": round(v3.get("total_tco2e_per_ha", 0.0), 1),
            # Absolute, matching Step 4's own presentation. The share of ceiling stays in the
            # per-AOI JSON under step 4 values, out of this table on purpose.
            "walker_unr_biomass_tco2e": (round(v4["unrealized_biomass_tco2e"])
                                         if v4.get("unrealized_biomass_tco2e") is not None
                                         else None),
            "walker_unr_combined_tco2e": round(v4.get("unrealized_combined_tco2e", 0.0)),
            "deferred_peat_ha": round(v3.get("deferred_peat_ha", 0.0), 1),
            "deferred_savanna_ha": round(v3.get("deferred_savanna_ha", 0.0), 1),
            "area_check": check_area_consistency(results),
            "status": "ok" if results["3"].applicable else "no ARR area",
        },
    }

In [ ]:
# Run the batch. One AOI failing does not stop the rest.
from IPython.display import display

runs: dict[str, dict] = {}
summary_rows: list[dict] = []

for i, path in enumerate(aoi_paths, start=1):
    print(f"\n[{i}/{len(aoi_paths)}] {path.stem}")
    try:
        rec = run_one(path)
    except Exception as exc:                      # one bad polygon must not lose the batch
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        summary_rows.append({"run_id": path.stem, "status": f"FAILED {type(exc).__name__}: {exc}"})
        continue

    runs[rec["run_id"]] = rec
    summary_rows.append(rec["row"])
    r3, r4 = rec["results"]["3"], rec["results"]["4"]
    print(f"  area {fmt_ha(rec['aoi'].area_ha)}, "
          f"Restore {fmt_ha(rec['results']['1'].values.get('restore_ha', 0.0))}, "
          f"quantified {fmt_ha(r3.values.get('quantified_ha', 0.0))}")
    print(f"  {r3.narrative}")
    if r4.applicable:
        print(f"  {r4.narrative}")
    print(f"  area check: {rec['row']['area_check']}   saved: {rec['json_path']}")

print(f"\nDone. {len(runs)} of {len(aoi_paths)} AOI(s) completed.")

---
## Summary across AOIs

One row per polygon. This table is for comparing sites; the per-AOI JSON stays the record of what
each number is made of.

Figures are absolute, like Step 4's own table: the ARR estimate and the Walker ceilings in tCO2e,
no percentage. The share of ceiling is in each AOI's JSON under step 4 `values` if it is wanted.

Flags are not repeated here. They are per AOI and they matter per AOI, so collapsing them into one
list would suggest they apply everywhere. `area_check` is the exception: it is a correctness
signal, not context, so every row carries it and any value other than `OK` means that row's carbon
total should not be reported until it is understood.

In [ ]:
summary = pd.DataFrame(summary_rows)
display(summary)

summary_path = Path(OUTPUT_ROOT) / f"summary__carbon-sequestration_{PROJECT_DURATION_YEARS}yr.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_path, index=False)
print("Saved", summary_path)

bad = [r for r in summary_rows if r.get("area_check") not in ("OK", "n/a", None)]
failed = [r for r in summary_rows if str(r.get("status", "")).startswith("FAILED")]
if bad:
    print(f"\n{len(bad)} AOI(s) failed the Step 2 vs Step 3 area check:",
          ", ".join(r["run_id"] for r in bad))
if failed:
    print(f"{len(failed)} AOI(s) did not run:", ", ".join(r["run_id"] for r in failed))
if not bad and not failed:
    print("\nAll AOIs ran and passed the area check.")

---
## Inspect one AOI

The batch prints one line per polygon. When a number needs explaining, set `INSPECT` to a run id
and this cell shows that AOI in full: the pathway table, the Restore categories with their carbon
status, the group breakdown, the annual curve, the Walker comparison, and every flag.

This reads the `runs` dictionary held in memory, so it needs the batch cell to have run in this
kernel. It reads nothing from disk and recomputes nothing.

In [ ]:
INSPECT = ""     # a run id from the summary; blank takes the first AOI that produced a number

from IPython.display import display

pick = INSPECT or next((k for k, v in runs.items() if v["results"]["3"].applicable), None)
if pick is None:
    print("Nothing to inspect: no AOI produced a carbon number.")
elif pick not in runs:
    print(f"'{pick}' is not in the completed runs. Available: {', '.join(runs) or 'none'}")
else:
    rec = runs[pick]
    print(f"=== {pick} ===  {fmt_ha(rec['aoi'].area_ha)}   {rec['json_path']}\n")
    for key in ("1", "2", "3", "4"):
        res = rec["results"][key]
        print(f"[{res.component}]" + ("" if res.applicable else "  (not applicable)"))
        if res.narrative:
            print(" ", res.narrative)
        for name, tbl in res.tables.items():
            if tbl:
                print(f"  {name}:")
                display(as_frame(tbl))
        for f in res.flags:
            print("FLAG:", f)
        print()
    print("area check:", rec["row"]["area_check"])